# Step 1 · Data and panel

**Purpose**: bring four kinds of public data (power outages, travel, business and spending, and the travel network) into one "event × county × day" panel for Step 2 (recovery curves) and Step 3 (network spillovers).

**Events**: 5 hurricanes (Laura, Zeta, Ida, Ian, Idalia) and 1 winter storm (Uri). Each event spans 35 days before day 0 to 42 days after, 78 days in all; days −35 to −8 are the baseline.

**Data**

| Data | Content | Events covered |
| --- | --- | --- |
| EAGLE-I | Customers without power by county, one record every 15 minutes | All 6 |
| BTS Trips by Distance | Daily trips and people leaving home, by county | All 6 (county data end on 6 April 2024) |
| Womply (Opportunity Insights) | Small-business revenue and number of businesses open, weekly, by county where the business is located | Laura, Zeta, Uri, Ida (data end in February 2022) |
| Affinity (Opportunity Insights) | Card spending, by county where the cardholder lives | Daily (7-day moving average) before June 2022, weekly after: Laura, Zeta, Uri and Ida get daily data, Ian and Idalia weekly |
| GeoDS | Weekly county-to-county flows of people, 8 weeks in September–October 2019, used to build the pre-pandemic dependency network | Shared by all 6 |

**Structure**

| Part | Content | Approx. time |
| --- | --- | --- |
| A | Events and study areas | 1 minute |
| B | Counties, population, customers | 1 minute |
| C | EAGLE-I outages (4 years, about 4.7 GB in all) | 15–30 minutes |
| D | BTS travel | 2–5 minutes |
| E | Business and spending | 1 minute |
| F | 2019 travel network (8 weeks, about 600 MB in all) | 3–6 minutes |
| G | Merge into the panel | 1–2 minutes |
| H | Coverage, two figures, checklist | 1 minute |

**Running the notebook**

- No Earth Engine is needed, only Google Drive
- Results go to a new Drive folder, `outage_mobility`. If the `us_flood_outage` folder of the companion outage-duration project is still there, part C uses its label table to check the outage figures, and the Census population file and EAGLE-I's `coverage_history.csv` are taken from it instead of being downloaded again
- Cells run in order from top to bottom. Each step's downloads and results are saved to Drive as soon as they are done, so a rerun after an interruption skips what is already finished

In [ ]:
# 1. Environment, folders, chart style
import os, io, time
from urllib.parse import quote
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    BASE, TMP = "/content/drive/MyDrive", "/content/tmp"
except ImportError:                     # in local Jupyter: use the current folder
    BASE, TMP = ".", "tmp"
OUT = f"{BASE}/outage_mobility"         # results of this project
P2  = f"{BASE}/us_flood_outage"         # the companion project's folder (may be absent)
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
print("Output folder:", OUT, "| companion project folder found:", os.path.isdir(P2))

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

def spearman(a, b):
    # Spearman rank correlation: rank first, then take the ordinary correlation
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank())

def as_day(s):
    # one date type (day, nanosecond unit) so dates from different sources can be merged
    return pd.to_datetime(s).dt.normalize().astype("datetime64[ns]")

# Chart style (shared by all figures). Outage-duration groups use one blue from light to dark; "no substantial outage" is a gray dashed line
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
GROUPS = [("none", "No outage of 10%+", "#7a7974", "--"),
          ("lt24", "Under 24 h",        "#86b6ef", "-"),
          ("24to72", "24–72 h",         "#2a78d6", "-"),
          ("ge72", "72 h or more",      "#104281", "-")]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## Parameters

After changing a parameter, rerun from the first part that uses it. Files already downloaded are not downloaded again.

In [ ]:
# 2. Parameters
PRE, POST = 35, 42                 # each event spans PRE days before day 0 to POST days after
BASE_FROM, BASE_TO = -35, -8       # baseline: days −35 to −8, leaving out the week of preparation and evacuation before landfall
MIN_BASE_DAYS = 2                  # each weekday needs at least this many baseline days before a change in travel is computed

STEP_H     = 0.25                  # one time slot = 15 minutes
OUT_THRESH = 0.10                  # "substantial outage": at least 10% of customers out (as in the companion outage-duration project)
CALM_HOURS = 24                    # restored = stays below the threshold for this many hours
MIN_PEAK   = 100                   # a county is "affected" only with at least this many customers out at the peak
GAP_FILL_H = 12                    # collection gaps up to this long are filled by interpolation; longer ones stay empty
MIN_DAY_COVER = 0.5                # a day's outage figures need data in at least half of its time slots

FLAG_BASE = 0.02                   # a baseline day with a mean outage share above 2%: the baseline is contaminated by another outage
FLAG_LATE = 0.05                   # a day after restoration with a mean outage share above 5% again: a second outage inside the window

NET_WEEKS = ["2019-09-09", "2019-09-16", "2019-09-23", "2019-09-30",     # the 8 weeks used for the network (the Monday of each week)
             "2019-10-07", "2019-10-14", "2019-10-21", "2019-10-28"]
BTS_PAGE = 50_000                  # the BTS interface returns at most this many rows per request

## A. Events and study areas

| Event | Day 0 | Study area (states) |
| --- | --- | --- |
| Laura 2020 | Landfall | LA, TX, AR, MS |
| Zeta 2020 | Landfall | LA, MS, AL, GA, SC, NC, VA |
| Uri 2021 | ERCOT begins rotating outages (15 February, 1:20 a.m. local time) | TX, OK, LA, AR, MS |
| Ida 2021 | Landfall | LA, MS, AL |
| Ian 2022 | Landfall | FL, GA, SC, NC |
| Idalia 2023 | Landfall | FL, GA, SC, NC, AL |

- Landfall times come from the official reports of the U.S. National Hurricane Center (NHC); Uri's time comes from the FERC and NERC inquiry. The `source` column links to each
- The study areas are wider than in the companion project: besides the states that were hit, they include neighboring states. Only then are there counties that barely lost power themselves but depend on hard-hit counties through travel, which Step 3 needs to test for spillovers
- Beryl and Helene (2024) are dropped, because BTS county travel data end on 6 April 2024. Idalia is new; it was not in the companion project
- **Day boundaries**: all daily data use calendar days in the event's time zone (U.S. Central or Eastern). Where a study area spans two time zones, counties in the other zone are off by one hour, which barely matters for daily figures

**Overlap with other storms**: Laura's baseline contains Hanna (25 July 2020, South Texas) and its later window contains Sally (16 September, Alabama); Zeta's baseline contains Delta (9 October, southwest Louisiana); Ida's window contains Nicholas (14 September 2021, Texas and Louisiana). Part C marks the counties concerned from the outage data themselves rather than from a hand-made list.

In [ ]:
# 3. Event table (t0 is in UTC, as in EAGLE-I)
STATES = {"01": "AL", "05": "AR", "12": "FL", "13": "GA", "22": "LA", "28": "MS",
          "37": "NC", "40": "OK", "45": "SC", "48": "TX", "51": "VA"}
NHC = "https://www.nhc.noaa.gov/data/tcr/"
FERC = ("https://www.ferc.gov/sites/default/files/2021-09/February%202021%20Cold%20Weather%20Grid%20Operations_"
        "Preliminary%20Findings%20and%20Recommendations-09-23-21_Full%20Presentation.pdf")
EVENTS = pd.DataFrame([
    dict(key="laura2020",  name="Laura",  kind="hurricane",    year=2020, t0="2020-08-27 06:00", place="Cameron, LA",
         cat="Cat 4", states=["22", "48", "05", "28"],                   tz="America/Chicago", source=NHC + "AL132020_Laura.pdf"),
    dict(key="zeta2020",   name="Zeta",   kind="hurricane",    year=2020, t0="2020-10-28 21:00", place="Cocodrie, LA",
         cat="Cat 3", states=["22", "28", "01", "13", "45", "37", "51"], tz="America/Chicago", source=NHC + "AL282020_Zeta.pdf"),
    dict(key="uri2021",    name="Uri",    kind="winter storm", year=2021, t0="2021-02-15 07:20", place="ERCOT load shed begins",
         cat="—",     states=["48", "40", "22", "05", "28"],             tz="America/Chicago", source=FERC),
    dict(key="ida2021",    name="Ida",    kind="hurricane",    year=2021, t0="2021-08-29 16:55", place="Port Fourchon, LA",
         cat="Cat 4", states=["22", "28", "01"],                         tz="America/Chicago", source=NHC + "AL092021_Ida.pdf"),
    dict(key="ian2022",    name="Ian",    kind="hurricane",    year=2022, t0="2022-09-28 19:05", place="Cayo Costa, FL",
         cat="Cat 4", states=["12", "13", "45", "37"],                   tz="America/New_York", source=NHC + "AL092022_Ian.pdf"),
    dict(key="idalia2023", name="Idalia", kind="hurricane",    year=2023, t0="2023-08-30 11:45", place="Keaton Beach, FL",
         cat="Cat 3", states=["12", "13", "45", "37", "01"],             tz="America/New_York", source=NHC + "AL102023_Idalia.pdf"),
]).set_index("key")
EVENTS["t0"] = pd.to_datetime(EVENTS["t0"])
EVENTS["day0"] = [t.tz_localize("UTC").tz_convert(tz).tz_localize(None).normalize()      # day 0 = the local date
                  for t, tz in zip(EVENTS["t0"], EVENTS["tz"])]
EVENTS["day0"] = EVENTS["day0"].astype("datetime64[ns]")
ALL_STATES = sorted({s for st in EVENTS["states"] for s in st})

def window_days(key):
    # every day of the event window (local dates)
    day0 = EVENTS.loc[key, "day0"]
    return pd.date_range(day0 - pd.Timedelta(days=PRE), day0 + pd.Timedelta(days=POST), freq="D").astype("datetime64[ns]")

def window_utc(key):
    # start and end of the window in UTC: local midnight of the first day to local midnight after the last day
    days, tz = window_days(key), EVENTS.loc[key, "tz"]
    to_utc = lambda d: d.tz_localize(tz).tz_convert("UTC").tz_localize(None)
    return to_utc(days[0]), to_utc(days[-1] + pd.Timedelta(days=1))

events_out = EVENTS.assign(states=[" ".join(STATES[s] for s in st) for st in EVENTS["states"]])
events_out.to_csv(f"{OUT}/step1_events.csv")
events_out[["name", "kind", "t0", "day0", "place", "cat", "states", "tz", "source"]]

## B. Counties, population, customers

The denominator of the outage share is the same as in the companion outage-duration project: **county customers = county population × statewide customers ÷ state population**. Population is the Census county estimate for the event's year; statewide customers come from EAGLE-I's `coverage_history.csv`, which only covers 2018–2022, so Idalia (2023) uses the 2022 figure.

The table below shows electricity customers per person in each state; it should normally lie between 0.4 and 0.6.

In [ ]:
# 4. Download helpers; county population; customers per county
def download(url, path, label="", min_bytes=1000):
    # download a file to path; skip it if it already exists
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return path
    print(f"Downloading {label or url} …")
    with requests.get(url, stream=True, timeout=180, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        done, shown = 0, 0
        with open(path + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block); done += len(block)
                if done - shown >= 400e6:
                    shown = done; print(f"  {done / 1e6:,.0f} MB downloaded")
    if done < min_bytes:
        os.remove(path + ".part")
        raise RuntimeError(f"The downloaded file is too small ({done} bytes); the site may have blocked the request. Open {url} in a browser, download it manually, "
                           f"upload it to the outage_mobility folder in Drive under the name {os.path.basename(path)}, then rerun this cell.")
    os.replace(path + ".part", path)
    return path

def get_file(sources, min_bytes=1000):
    # sources is a list of (address, file name). Look for a file of that name in this project's and the companion project's folders first, otherwise download in turn
    for url, name in sources:
        for folder in (OUT, P2):
            p = f"{folder}/{name}"
            if os.path.exists(p) and os.path.getsize(p) >= min_bytes:
                return p
    errors = []
    for url, name in sources:
        try:
            return download(url, f"{OUT}/{name}", name, min_bytes)
        except Exception as err:
            errors.append(f"{name}: {err}")
    raise RuntimeError("\n".join(errors))

CENSUS = "https://www2.census.gov/programs-surveys/popest/datasets"
CENSUS_SOURCES = [(f"{CENSUS}/2020-2024/counties/totals/co-est2024-alldata.csv", "co-est2024-alldata.csv"),
                  (f"{CENSUS}/2020-2023/counties/totals/co-est2023-alldata.csv", "co-est2023-alldata.csv")]
COVERAGE_SOURCES = [("https://ndownloader.figshare.com/files/42547714", "coverage_history.csv")]

c = pd.read_csv(get_file(CENSUS_SOURCES, 100_000), encoding="latin-1", dtype={"STATE": str, "COUNTY": str})
c = c[c["SUMLEV"] == 50].copy()
c.index = c["STATE"].str.zfill(2) + c["COUNTY"].str.zfill(3)
years = {col: int(col[-4:]) for col in c.columns if col.startswith("POPESTIMATE") and col[-4:].isdigit()}
pop = c[list(years)].rename(columns=years)
pop = pop[pop.index.str[:2].isin(ALL_STATES)].dropna()                # rows = counties, columns = years
county_name = (c["CTYNAME"].reindex(pop.index).str.replace(" County", "", regex=False)
               .str.replace(" Parish", "", regex=False) + ", " + pop.index.str[:2].map(STATES))
print(f"Population: {len(pop)} counties, years {min(pop.columns)}–{max(pop.columns)}")

def pop_year(year):
    # the year in the population table closest to this one
    return min(pop.columns, key=lambda y: abs(y - year))

cov = pd.read_csv(get_file(COVERAGE_SOURCES), encoding="utf-8-sig")
cov.columns = [x.strip().lower() for x in cov.columns]
txt = cov["year"].astype(str)                                      # the year is written "1/1/18" or "2018"; both become 2018
yr = pd.to_numeric(txt.str.extract(r"((?:19|20)\d\d)")[0], errors="coerce")
yr = yr.fillna(2000 + pd.to_numeric(txt.str.extract(r"(\d{1,2})\s*$")[0], errors="coerce"))
cov["year"] = yr.astype(int)
cov["state"] = cov["state"].astype(str).str.strip()
cov["total_customers"] = pd.to_numeric(cov["total_customers"].astype(str).str.replace(",", "", regex=False), errors="coerce")
REF_MIN, REF_MAX = int(cov["year"].min()), int(cov["year"].max())

def customers_per_person(state, year):
    # statewide customers ÷ state population, from the nearest year that has data
    ref = min(max(year, REF_MIN), REF_MAX)
    row = cov[(cov["state"] == STATES[state]) & (cov["year"] == ref)]
    if row.empty:
        raise ValueError(f"coverage_history.csv has no record for {STATES[state]} in {ref}")
    return row["total_customers"].iloc[0] / pop.loc[pop.index.str[:2] == state, pop_year(ref)].sum()

den = {}                                                           # den[event] = customers in each county of the study area
for key, ev in EVENTS.iterrows():
    parts = [pop.loc[pop.index.str[:2] == s, pop_year(ev["year"])] * customers_per_person(s, ev["year"]) for s in ev["states"]]
    den[key] = pd.concat(parts).round().sort_index()
    den[key].index.name = "fips"

ratio = pd.DataFrame({y: {STATES[s]: customers_per_person(s, y) for s in ALL_STATES} for y in sorted(set(EVENTS["year"]))})
print("Customers per person in each state:")
display(tidy(ratio, 3))
pd.Series({k: len(v) for k, v in den.items()}, name="counties in the study area").to_frame().T

## C. EAGLE-I outages

As in the companion outage-duration project: EAGLE-I records customers out in each county every 15 minutes (in UTC) and **does not record zeros**. When a state has no record at all at some time, that is treated as a collection gap and every county of the state is left empty at that time; gaps of up to 12 hours are filled by interpolation, longer ones stay empty. This version does **not** apply the companion project's "sharp drop" correction, because it barely changes daily averages; the end of part C checks this against the companion project's label table.

Each county gets two kinds of results:

**Daily figures** (`step1_outage_daily.csv.gz`)

| Column | Meaning |
| --- | --- |
| `out_mean_pct` | Mean outage share over the day (%) |
| `out_max_pct` | Highest outage share during the day (%) |
| `out_cust_hours` | Customer-hours without power that day |
| `out_cover` | Share of the day's time slots with data. On days below 50%, the three columns above are left empty |

**Event labels** (`step1_outage_event.csv`), defined as in the companion outage-duration project and computed only from 2 days before day 0 onward:

| Column | Meaning |
| --- | --- |
| `peak_pct` | Peak outage share (%) between 2 days before and 7 days after day 0 |
| `hours_above` | Cumulative hours with an outage share ≥ 10%, counted up to restoration |
| `days_to_restore` | Days after day 0 until restoration (the first time after the peak that stays below 10% for 24 hours) |
| `affected` / `censored` | Affected county / affected but not restored within the window |
| `flag_base` | Some baseline day had a mean outage share above 2%: the county's baseline is contaminated by another outage, and Step 2 leaves it out by default |
| `flag_late` | After restoration, some day had a mean outage share above 5% again: a second outage late in the window |

In [ ]:
# 5. Download EAGLE-I year by year and keep each event's states and time span
EAGLEI_FILES = {2020: 42547894, 2021: 42547891, 2022: 42547897, 2023: 44574907}    # figshare file ids

def parse_time(s):
    # the two timestamp styles that may appear in the files
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%y %H:%M", None):
        t = pd.to_datetime(s, format=fmt, errors="coerce")
        if t.notna().mean() > 0.99: break
    return t

def cache_path(key):
    return f"{OUT}/eaglei_{key}.csv.gz"

for year, file_id in EAGLEI_FILES.items():
    todo = [k for k in EVENTS.index[EVENTS["year"] == year] if not os.path.exists(cache_path(k))]
    if not todo:
        continue
    raw = next((p for p in (f"{OUT}/eaglei_outages_{year}.csv", f"{P2}/eaglei_outages_{year}.csv") if os.path.exists(p)), None)
    if raw is None:                                         # a file downloaded by hand can be placed in the outage_mobility folder
        raw = download(f"https://ndownloader.figshare.com/files/{file_id}", f"{TMP}/eaglei_outages_{year}.csv",
                       f"EAGLE-I {year} (about 1.2 GB)", 300e6)
    header = {x.strip().lower(): x for x in pd.read_csv(raw, nrows=0, encoding="utf-8-sig").columns}
    fips_col, time_col = header["fips_code"], header["run_start_time"]
    out_col = header.get("customers_out", header.get("sum"))
    if out_col is None:
        raise KeyError(f"No customers-out column in the {year} file; the header is: {list(header.values())}")
    states = sorted({s for k in todo for s in EVENTS.loc[k, "states"]})
    parts = {k: [] for k in todo}
    for chunk in pd.read_csv(raw, usecols=[fips_col, out_col, time_col], dtype={fips_col: str},
                             encoding="utf-8-sig", chunksize=2_000_000):
        fips = chunk[fips_col].str.split(".").str[0].str.zfill(5)
        keep = fips.str[:2].isin(states)
        if not keep.any():
            continue
        sub = pd.DataFrame({"fips": fips[keep], "customers_out": pd.to_numeric(chunk.loc[keep, out_col], errors="coerce"),
                            "time": parse_time(chunk.loc[keep, time_col])}).dropna()
        for k in todo:
            t0, t1 = window_utc(k)
            m = sub["fips"].str[:2].isin(EVENTS.loc[k, "states"]) & (sub["time"] >= t0) & (sub["time"] < t1)
            if m.any():
                parts[k].append(sub[m])
    for k in todo:
        one = pd.concat(parts[k]) if parts[k] else pd.DataFrame(columns=["fips", "customers_out", "time"])
        one.to_csv(cache_path(k), index=False)
    if raw.startswith(TMP):
        os.remove(raw)                                      # delete the large raw file once used, to save Colab disk space
    print(f"{year}: extracted {', '.join(todo)}")

ea = {}
for k in EVENTS.index:
    ea[k] = pd.read_csv(cache_path(k), dtype={"fips": str})
    ea[k]["time"] = pd.to_datetime(ea[k]["time"])
pd.DataFrame({k: {"rows": len(d), "counties_with_records": d["fips"].nunique(),
                  "first": d["time"].min(), "last": d["time"].max()} for k, d in ea.items()}).T

In [ ]:
# 6. Each event: 15-minute panel → daily outage figures and event labels
CALM_SLOTS = int(CALM_HOURS / STEP_H)

def run_lengths(flag):
    # for each True, the length of the run of consecutive Trues it belongs to (0 where False)
    flag = pd.Series(np.asarray(flag, dtype=bool))
    return flag.groupby((~flag).cumsum()).transform("sum").where(flag, 0).values

def first_calm(below, after_peak):
    # the first time after the peak from which the condition holds for the next CALM_HOURS hours; NaT if there is none
    calm = below.astype(float)[::-1].rolling(CALM_SLOTS, min_periods=CALM_SLOTS).min()[::-1].eq(1) & after_peak
    return calm.idxmax().where(calm.any())

def to_long(frame, name):
    # wide table (rows = dates, columns = counties) → long table
    return frame.rename_axis(index="date", columns="fips").reset_index().melt(id_vars="date", var_name="fips", value_name=name)

def outage_event(key):
    ev, d = EVENTS.loc[key], den[key]
    cols = d.index
    state_of = cols.str[:2]
    t0u, t1u = window_utc(key)
    grid = pd.date_range(t0u, t1u, freq="15min", inclusive="left")
    e = ea[key].assign(slot=ea[key]["time"].dt.floor("15min"))
    wide = (e.groupby(["slot", "fips"])["customers_out"].max().unstack("fips")
              .reindex(index=grid, columns=cols).fillna(0.0))

    # whole-state gaps: the state has no record at all in this slot
    gap = pd.DataFrame(False, index=grid, columns=cols)
    for s in ev["states"]:
        has = grid.isin(e.loc[e["fips"].str[:2] == s, "slot"])
        gap.loc[~has, state_of == s] = True
    wide = wide.mask(gap)
    # fill short gaps: interpolate everything, then empty the long gaps again
    limit = int(GAP_FILL_H / STEP_H)
    long_gap = pd.DataFrame(False, index=grid, columns=cols)
    for s in ev["states"]:
        g = gap.loc[:, state_of == s].iloc[:, 0]
        long_gap.loc[run_lengths(g) > limit, state_of == s] = True
    wide = wide.interpolate(method="time", limit_area="inside").mask(long_gap)
    share = wide.div(d, axis=1).clip(upper=1.0)

    # daily figures (calendar days in the event's time zone)
    day = pd.DatetimeIndex(grid.tz_localize("UTC").tz_convert(ev["tz"]).tz_localize(None).normalize()).astype("datetime64[ns]")
    slots = pd.Series(1, index=grid).groupby(day).sum()
    cover = share.notna().groupby(day).sum().div(slots, axis=0)
    ok = cover >= MIN_DAY_COVER
    mean_pct = share.groupby(day).mean().where(ok) * 100
    daily = to_long(mean_pct, "out_mean_pct")
    daily["out_max_pct"] = to_long(share.groupby(day).max().where(ok) * 100, "v")["v"].values
    daily["out_cust_hours"] = to_long(wide.groupby(day).mean().where(ok) * 24, "v")["v"].values
    daily["out_cover"] = to_long(cover, "v")["v"].values
    daily.insert(0, "event", key)

    # event labels (same definitions as the companion project, counted only from 2 days before day 0)
    t0 = ev["t0"]
    since = grid >= t0 - pd.Timedelta(days=2)
    zone = since & (grid <= t0 + pd.Timedelta(days=7))
    peak_out = wide[zone].max()
    peak_time = wide[zone].fillna(0).idxmax()
    after_peak = pd.DataFrame(grid.values[:, None] >= peak_time.values[None, :], index=grid, columns=cols)
    above = share >= OUT_THRESH
    restored = first_calm(share < OUT_THRESH, after_peak)
    end = grid[-1] + pd.Timedelta(minutes=15)
    until = pd.DataFrame(grid.values[:, None] < restored.fillna(end).values[None, :], index=grid, columns=cols)
    counted = until.copy()
    counted.loc[~since] = False                                    # nothing before 2 days ahead of day 0 is counted
    lab = pd.DataFrame({
        "event": key, "fips": cols, "name": county_name.reindex(cols).values, "state": state_of.map(STATES),
        "customers_est": d.values, "has_records": cols.isin(e["fips"].unique()),
        "peak_customers_out": peak_out.values, "peak_pct": share[zone].max().values * 100,
        "peak_time": peak_time.values, "restored_time": restored.values,
        "hours_above": (above & counted).sum().values * STEP_H,
        "days_to_restore": ((restored - t0) / pd.Timedelta(days=1)).values,
        "customer_hours": wide.where(counted).sum().values * STEP_H,
        "missing_storm_h": (wide.isna() & counted).sum().values * STEP_H,
    })
    lab["affected"] = (lab["peak_pct"] >= OUT_THRESH * 100) & (lab["peak_customers_out"] >= MIN_PEAK)
    lab["censored"] = lab["affected"] & lab["restored_time"].isna()

    # two flags: another outage in the baseline; a new outage after restoration
    rel = np.asarray((mean_pct.index - ev["day0"]).days)
    lab["base_outage_max_pct"] = mean_pct[(rel >= BASE_FROM) & (rel <= BASE_TO)].max().values
    start = np.where(lab["affected"] & lab["restored_time"].notna(), np.ceil(lab["days_to_restore"]) + 3,
                     np.where(lab["affected"], np.nan, 8.0))         # from the 3rd day after restoration; unaffected counties from day 8
    lab["late_outage_max_pct"] = mean_pct.where(rel[:, None] >= start[None, :]).max().values
    lab["flag_base"] = lab["base_outage_max_pct"] >= FLAG_BASE * 100
    lab["flag_late"] = lab["late_outage_max_pct"] >= FLAG_LATE * 100

    quality = {"filled_gap_h": float((gap & ~long_gap).any(axis=1).sum() * STEP_H),
               "long_gap_h": float(long_gap.any(axis=1).sum() * STEP_H),
               "long_gap_peak_h": float(long_gap[zone].any(axis=1).sum() * STEP_H)}
    return wide, daily, lab, quality

panels, daily_parts, label_parts, quality = {}, [], [], {}
for key in EVENTS.index:
    panels[key], dl, lb, quality[key] = outage_event(key)
    daily_parts.append(dl); label_parts.append(lb)
outage_daily = pd.concat(daily_parts, ignore_index=True)
labels = pd.concat(label_parts, ignore_index=True)
outage_daily.to_csv(f"{OUT}/step1_outage_daily.csv.gz", index=False)
labels.to_csv(f"{OUT}/step1_outage_event.csv", index=False)

A = labels[labels["affected"]]
summary = pd.DataFrame({
    "counties": labels.groupby("event").size(),
    "with_records": labels.groupby("event")["has_records"].sum(),
    "affected": A.groupby("event").size(),
    "over_24h": A[A["hours_above"] >= 24].groupby("event").size(),
    "over_72h": A[A["hours_above"] >= 72].groupby("event").size(),
    "censored": A.groupby("event")["censored"].sum(),
    "flag_base": A.groupby("event")["flag_base"].sum(),
    "flag_late": A.groupby("event")["flag_late"].sum(),
}).reindex(EVENTS.index).fillna(0).astype(int).join(pd.DataFrame(quality).T)
summary

In [ ]:
# 7. Check against the companion project's label table (skipped if the companion project's folder is absent)
p2_labels = next((p for p in (f"{P2}/step2c_labels_county_event.csv", f"{P2}/step2_labels_county_event.csv") if os.path.exists(p)), None)
validation = None
if p2_labels is None:
    print("The companion project's label table was not found; the check is skipped.")
else:
    old = pd.read_csv(p2_labels, dtype={"fips": str})
    old["affected"] = old["affected"].astype(str).str.lower().eq("true")
    m = labels.merge(old[["event", "fips", "hours_above", "peak_pct", "affected"]], on=["event", "fips"], suffixes=("", "_p2"))
    both = m[m["affected"] & m["affected_p2"]]
    rows = []
    for key, g in both.groupby("event"):
        rows.append({"event": key, "counties_both_affected": len(g),
                     "affected_here_only": int((m["event"].eq(key) & m["affected"] & ~m["affected_p2"]).sum()),
                     "affected_p2_only": int((m["event"].eq(key) & ~m["affected"] & m["affected_p2"]).sum()),
                     "spearman_hours_above": spearman(g["hours_above"], g["hours_above_p2"]),
                     "spearman_peak_pct": spearman(g["peak_pct"], g["peak_pct_p2"]),
                     "median_abs_diff_hours": (g["hours_above"] - g["hours_above_p2"]).abs().median()})
    validation = pd.DataFrame(rows).set_index("event")
    validation.to_csv(f"{OUT}/step1_validation_p2.csv")
    print(f"Compared with: {p2_labels}")
    display(tidy(validation, 3))

## D. BTS travel

The Bureau of Transportation Statistics (BTS) Trips by Distance data: daily trips, people staying home and people leaving home in each county, estimated by the University of Maryland from several sources of phone location data. Note:

- For a county with fewer than 50 devices in the sample on a given day, BTS does not publish that day's figures
- County data end on 6 April 2024
- This cell downloads through BTS's public interface, by "event × state", and saves each event as `bts_<event>.csv.gz`

Part G compares each day with the same county's average for **the same weekday** in the baseline, which gives two results: `trips_change_pct` (change in trips, %) and `not_home_change_pp` (change in the share of people leaving home, percentage points).

In [ ]:
# 8. Download BTS daily county travel
BTS_URL  = "https://data.bts.gov/resource/w96p-f2qv.csv"
BTS_COLS = ["date", "county_fips", "pop_stay_at_home", "pop_not_stay_at_home", "trips"]

def bts_get(params):
    # one request; on failure wait and retry, up to 4 times
    for attempt in range(4):
        try:
            r = requests.get(BTS_URL, params=params, timeout=180, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            return pd.read_csv(io.StringIO(r.text), dtype=str)
        except Exception as err:
            if attempt == 3:
                raise RuntimeError(f"The BTS interface failed 4 times in a row: {err}. Rerun this cell in a few minutes; events already downloaded are kept.")
            time.sleep(10 * (attempt + 1))

def bts_state(abbr, first, last):
    # all county records of one state for a range of dates (paging when needed)
    parts, offset = [], 0
    while True:
        part = bts_get({"$select": ",".join(BTS_COLS),
                        "$where": (f"level='County' AND state_code='{abbr}' AND date between "
                                   f"'{first:%Y-%m-%d}T00:00:00' and '{last:%Y-%m-%d}T00:00:00'"),
                        "$order": "county_fips,date", "$limit": BTS_PAGE, "$offset": offset})
        parts.append(part)
        if len(part) < BTS_PAGE:
            break
        offset += BTS_PAGE
    return pd.concat(parts, ignore_index=True)

mob = {}
for key in EVENTS.index:
    path = f"{OUT}/bts_{key}.csv.gz"
    if not os.path.exists(path):
        days = window_days(key)
        got = pd.concat([bts_state(STATES[s], days[0], days[-1]) for s in EVENTS.loc[key, "states"]], ignore_index=True)
        got.to_csv(path, index=False)
        print(f"{key}: downloaded {len(got):,} rows")
    m = pd.read_csv(path, dtype={"county_fips": str, "date": str})
    m["fips"] = m["county_fips"].str.split(".").str[0].str.zfill(5)
    m["date"] = as_day(m["date"].str[:10])
    for col in ["pop_stay_at_home", "pop_not_stay_at_home", "trips"]:
        m[col] = pd.to_numeric(m[col], errors="coerce")
    m["pop_total"] = m["pop_stay_at_home"] + m["pop_not_stay_at_home"]
    m["share_not_home"] = m["pop_not_stay_at_home"] / m["pop_total"]
    mob[key] = m[["fips", "date", "trips", "pop_total", "share_not_home"]].drop_duplicates(["fips", "date"])

pd.DataFrame({k: {"rows": len(m), "counties": m["fips"].nunique(), "first": m["date"].min(), "last": m["date"].max(),
                  "days_with_trips_pct": m["trips"].notna().sum() / (len(den[k]) * len(window_days(k))) * 100}
              for k, m in mob.items()}).T

## E. Business and spending (Opportunity Insights)

| Series | Source | Placed by | Frequency | Period |
| --- | --- | --- | --- | --- |
| `revenue`, `merchants` | Womply: small-business revenue and number of small businesses open | County where the business is | Weekly (weeks end on Sunday) | January 2020–February 2022 |
| `spend` | Affinity: credit and debit card spending | County where the cardholder lives | Daily (7-day moving average) until 5 June 2022, weekly after | 2019–June 2024 |

Notes:

- Values are "change since January 2020", seasonally adjusted. For example, −0.1 means 10% below January 2020. Here the baseline average is subtracted as well, giving `change_pp`: the change from the baseline, in percentage points of the January 2020 level
- A daily or weekly value reflects the 7 days up to its date, so the baseline only uses values whose 7 days all fall inside the baseline (days −29 to −8)
- The county files only have totals, without breakdowns by income. In Louisiana Womply covers only 12 parishes and in Mississippi 6 counties, mostly urban ones

In [ ]:
# 9. County data from Womply and Affinity
OI_URL = "https://raw.githubusercontent.com/OpportunityInsights/EconomicTracker/main/data/"

aff = pd.read_csv(download(OI_URL + quote("Affinity - County - Daily.csv"), f"{OUT}/oi_affinity_county.csv",
                           "Opportunity Insights: Affinity - County - Daily", 1_000_000), dtype={"spend_all": str})
wom = pd.read_csv(download(OI_URL + quote("Womply - County - Weekly.csv"), f"{OUT}/oi_womply_county.csv",
                           "Opportunity Insights: Womply - County - Weekly", 100_000))
aff["date"] = as_day(pd.to_datetime(aff[["year", "month", "day"]]))
wom["date"] = as_day(pd.to_datetime(wom[["year", "month", "day_endofweek"]].rename(columns={"day_endofweek": "day"})))
for d in (aff, wom):
    d["fips"] = d["countyfips"].astype(int).astype(str).str.zfill(5)
series = pd.concat([
    pd.DataFrame({"fips": aff["fips"], "date": aff["date"], "series": "spend", "freq": aff["freq"],
                  "value": pd.to_numeric(aff["spend_all"], errors="coerce")}),             # "." marks a masked value
    pd.DataFrame({"fips": wom["fips"], "date": wom["date"], "series": "revenue", "freq": "w", "value": wom["revenue_all"]}),
    pd.DataFrame({"fips": wom["fips"], "date": wom["date"], "series": "merchants", "freq": "w", "value": wom["merchants_all"]}),
], ignore_index=True).dropna(subset=["value"])

parts = []
for key, ev in EVENTS.iterrows():
    s = series[series["fips"].isin(den[key].index)].copy()
    s["rel_day"] = (s["date"] - ev["day0"]).dt.days
    s = s[s["rel_day"].between(BASE_FROM, POST + 6)]
    s.insert(0, "event", key)
    parts.append(s)
biz = pd.concat(parts, ignore_index=True)
inb = biz["rel_day"].between(BASE_FROM + 6, BASE_TO)                # all 7 days inside the baseline
base = biz[inb].groupby(["event", "fips", "series"])["value"].agg(base="mean", base_n="count").reset_index()
biz = biz.merge(base, on=["event", "fips", "series"], how="left")
biz["change_pp"] = ((biz["value"] - biz["base"]) * 100).where(biz["base_n"] >= 2)
biz.to_csv(f"{OUT}/step1_business_panel.csv.gz", index=False)

cov_biz = (biz.dropna(subset=["change_pp"]).groupby(["event", "series"])
              .agg(counties=("fips", "nunique"), freq=("freq", lambda f: "/".join(sorted(set(f)))))
              .unstack("series").reindex(EVENTS.index))
cov_biz

## F. The 2019 travel network (GeoDS)

The GeoDS Lab (University of Wisconsin–Madison, Song Gao's group) computed weekly county-to-county flows of people from SafeGraph phone data and published them on GitHub. Here the 8 weeks from 9 September to 3 November 2019 are summed into the pre-pandemic dependency network. Only flows that start or end inside a study area are kept.

- `visitor_flows`: visits observed by SafeGraph; `pop_flows`: an estimate scaled up to the population. Step 3 uses `visitor_flows` by default
- Each county's flows to itself are in the table as well; they are needed to compute how much a county depends on other counties
- Tropical Storm Imelda (17–19 September 2019, Houston) falls in these 8 weeks. Summed over 8 weeks its effect is small, but it could affect results around Houston in Step 3

In [ ]:
# 10. Download 8 weeks of 2019 county-to-county flows and sum them into the network
GEODS_URL = ("https://raw.githubusercontent.com/GeoDS/COVID19USFlows-WeeklyFlows/master/weekly_flows/"
              "county2county/weekly_county2county_{}.csv")
NET_PATH = f"{OUT}/step1_network_2019.csv.gz"

if not os.path.exists(NET_PATH):
    acc = []
    for wk in NET_WEEKS:
        tag = wk.replace("-", "_")
        p = download(GEODS_URL.format(tag), f"{TMP}/geods_{tag}.csv", f"GeoDS {wk}", 10e6)
        f = pd.read_csv(p, usecols=["geoid_o", "geoid_d", "visitor_flows", "pop_flows"], dtype={"geoid_o": str, "geoid_d": str})
        f["geoid_o"], f["geoid_d"] = f["geoid_o"].str.zfill(5), f["geoid_d"].str.zfill(5)
        acc.append(f[f["geoid_o"].str[:2].isin(ALL_STATES) | f["geoid_d"].str[:2].isin(ALL_STATES)])
        os.remove(p)
    net = (pd.concat(acc, ignore_index=True).groupby(["geoid_o", "geoid_d"], as_index=False)
             .agg(visitor_flows=("visitor_flows", "sum"), pop_flows=("pop_flows", "sum"), weeks=("visitor_flows", "size")))
    net.to_csv(NET_PATH, index=False)
net = pd.read_csv(NET_PATH, dtype={"geoid_o": str, "geoid_d": str})

out_tot = net.groupby("geoid_o")["visitor_flows"].sum()
self_flow = net[net["geoid_o"] == net["geoid_d"]].set_index("geoid_o")["visitor_flows"]
stay = (self_flow / out_tot).reindex(out_tot.index)
region = sorted(set().union(*[set(v.index) for v in den.values()]))
print(f"Network: {len(net):,} county pairs; of the {len(region)} counties in the study areas, {out_tot.index.isin(region).sum()} appear as an origin")
print(f"Share of visits that stay in the home county (median over study-area counties): {stay.reindex(region).median():.2f}")

## G. Merge into the panel

Each row is "event × county × day"; every county of the study area has a row for every day of the window, with blanks where data are missing. The main columns:

| Column | Meaning |
| --- | --- |
| `rel_day` | Days since day 0 |
| `out_mean_pct` etc. | Daily outage figures from part C |
| `trips`, `share_not_home` | BTS trips and share of people leaving home |
| `base_trips`, `base_not_home`, `base_n` | The same county's average for the same weekday in the baseline, and how many days it uses |
| `trips_change_pct` | Change in trips from the baseline (%) |
| `not_home_change_pp` | Change in the share of people leaving home from the baseline (percentage points) |

Business and spending are mostly weekly, so they are kept separately in `step1_business_panel.csv.gz`.

In [ ]:
# 11. The event × county × day panel
frames = []
for key in EVENTS.index:
    g = pd.MultiIndex.from_product([den[key].index, window_days(key)], names=["fips", "date"]).to_frame(index=False)
    g.insert(0, "event", key)
    frames.append(g)
panel = pd.concat(frames, ignore_index=True)
panel["date"] = as_day(panel["date"])
panel["state"] = panel["fips"].str[:2].map(STATES)
panel["rel_day"] = (panel["date"] - panel["event"].map(EVENTS["day0"])).dt.days
panel["weekday"] = panel["date"].dt.dayofweek

od = outage_daily.assign(date=as_day(outage_daily["date"]))
panel = panel.merge(od, on=["event", "fips", "date"], how="left")
mob_all = pd.concat([m.assign(event=k) for k, m in mob.items()], ignore_index=True)
panel = panel.merge(mob_all, on=["event", "fips", "date"], how="left")

inb = panel["rel_day"].between(BASE_FROM, BASE_TO)
base = (panel[inb].groupby(["event", "fips", "weekday"])
          .agg(base_trips=("trips", "mean"), base_not_home=("share_not_home", "mean"), base_n=("trips", "count"))
          .reset_index())
panel = panel.merge(base, on=["event", "fips", "weekday"], how="left")
ok = (panel["base_n"] >= MIN_BASE_DAYS) & (panel["base_trips"] > 0)
panel["trips_change_pct"] = ((panel["trips"] / panel["base_trips"] - 1) * 100).where(ok)
panel["not_home_change_pp"] = ((panel["share_not_home"] - panel["base_not_home"]) * 100).where(ok)
panel.to_csv(f"{OUT}/step1_panel_daily.csv.gz", index=False)

print(f"Panel: {len(panel):,} rows, {panel['fips'].nunique()} counties, {panel['event'].nunique()} events")
tidy(panel.groupby("event")[["out_mean_pct", "trips", "trips_change_pct", "not_home_change_pp"]]
          .agg(lambda x: x.notna().mean() * 100).reindex(EVENTS.index)
          .rename(columns=lambda c: c + "_filled_pct"))

## H. Coverage, two figures, checklist

First the counties of each event are put into four groups by outage duration: no substantial outage (peak below 10%), under 24 hours, 24–72 hours, and 72 hours or more. The coverage table and both figures below use these groups. Counties with `flag_base` (baseline contaminated by another outage) are left out of the figures.

In [ ]:
# 12. Outage groups and the coverage table
labels["group"] = np.select([~labels["affected"], labels["hours_above"] < 24, labels["hours_above"] < 72],
                            ["none", "lt24", "24to72"], "ge72")
days_n = {k: len(window_days(k)) for k in EVENTS.index}
mob_days = (panel.dropna(subset=["trips"]).groupby(["event", "fips"]).size()
                 .div(panel.groupby(["event", "fips"]).size()).rename("mob_days_share"))
L = labels.merge(mob_days.reset_index(), on=["event", "fips"], how="left").fillna({"mob_days_share": 0})
L["has_mob"] = L["mob_days_share"] >= 0.9
biz_cty = biz.dropna(subset=["change_pp"]).groupby(["event", "series"])["fips"].unique()
net_origins = set(net["geoid_o"])

rows = []
for key in EVENTS.index:
    l = L[L["event"] == key]
    a24 = l[l["affected"] & (l["hours_above"] >= 24)]
    def biz_n(series, sub=l):
        got = biz_cty.get((key, series), [])
        return int(sub["fips"].isin(got).sum())
    rows.append({"event": key, "counties": len(l), "with_eaglei": int(l["has_records"].sum()),
                 "affected": int(l["affected"].sum()), "over_24h": len(a24), "over_72h": int((l["group"] == "ge72").sum()),
                 "mob_counties": int(l["has_mob"].sum()),
                 "mob_share_over_24h": a24["has_mob"].mean() if len(a24) else np.nan,
                 "flag_base_affected": int((l["affected"] & l["flag_base"]).sum()),
                 "flag_base_share_affected": (l.loc[l["affected"], "flag_base"].mean() if l["affected"].any() else np.nan),
                 "flag_late_affected": int((l["affected"] & l["flag_late"]).sum()),
                 "revenue_counties": biz_n("revenue"), "revenue_over_24h": biz_n("revenue", a24),
                 "spend_counties": biz_n("spend"), "spend_over_24h": biz_n("spend", a24),
                 "network_share": l["fips"].isin(net_origins).mean(),
                 "long_gap_peak_h": quality[key]["long_gap_peak_h"]})
coverage = pd.DataFrame(rows).set_index("event")
coverage.to_csv(f"{OUT}/step1_coverage.csv")
labels.to_csv(f"{OUT}/step1_outage_event.csv", index=False)                    # save the groups into the label table too
tidy(coverage, 2)

In [ ]:
# 13. Figure 1: change in trips by outage group (median county), by days since day 0
MIN_GROUP = 3                                                                  # groups with fewer than 3 counties are not drawn
use = panel.merge(labels[["event", "fips", "group", "flag_base"]], on=["event", "fips"])
use = use[~use["flag_base"] & use["rel_day"].between(-14, POST)]
fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.6), sharey=True)
for ax, key in zip(axes.flat, EVENTS.index):
    sub = use[use["event"] == key]
    ax.axvline(0, color=FAINT, lw=1); ax.axhline(0, color=FAINT, lw=1)
    for g, label, color, ls in GROUPS:
        s = sub[sub["group"] == g]
        n = s.dropna(subset=["trips_change_pct"])["fips"].nunique()
        if n < MIN_GROUP:
            continue
        med = s.groupby("rel_day")["trips_change_pct"].median()
        ax.plot(med.index, med.values, color=color, lw=2, ls=ls, label=f"{label} ({n})")
    ev = EVENTS.loc[key]
    ax.set_title(f"{ev['name']} {ev['year']}")
    ax.set_xticks([-14, -7, 0, 7, 14, 21, 28, 35, 42])
    ax.legend(fontsize=8, loc="lower right", title="Outage duration (counties)", title_fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("Days since day 0")
for ax in axes[:, 0]:
    ax.set_ylabel("Change in daily trips (%)")
fig.suptitle("Daily trips against the same weekday in the baseline, median county by outage duration",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step1_fig1_trips_by_outage.png"); plt.show()

In [ ]:
# 14. Figure 2: business revenue (Womply, where businesses are) and card spending (Affinity, where people live)
bz = biz.merge(labels[["event", "fips", "group", "flag_base"]], on=["event", "fips"])
bz = bz[~bz["flag_base"] & bz["rel_day"].between(-14, POST + 6)]
rows_ = [("revenue", "Small-business revenue"), ("spend", "Card spending")]
fig, axes = plt.subplots(2, len(EVENTS), figsize=(16, 6.8), sharey="row", sharex=True)
counts = {}
for i, (series_, row_label) in enumerate(rows_):
    for j, key in enumerate(EVENTS.index):
        ax = axes[i, j]
        sub = bz[(bz["event"] == key) & (bz["series"] == series_)]
        ax.axvline(0, color=FAINT, lw=1); ax.axhline(0, color=FAINT, lw=1)
        drawn = 0
        for g, label, color, ls in GROUPS:
            s = sub[sub["group"] == g].dropna(subset=["change_pp"])
            n = s["fips"].nunique()
            if n < MIN_GROUP:
                continue
            med = s.groupby("rel_day")["change_pp"].median()
            weekly = (s["freq"] == "w").all()
            ax.plot(med.index, med.values, color=color, lw=2, ls=ls, marker="o" if weekly else None, ms=4)
            counts[(key, series_, label)] = n
            drawn += 1
        if drawn == 0:
            ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha="center", va="center", color=MUTED)
        if i == 0:
            ev = EVENTS.loc[key]; ax.set_title(f"{ev['name']} {ev['year']}")
        if j == 0:
            ax.set_ylabel(f"{row_label}\n(change, pp of Jan 2020)")
        if i == 1:
            ax.set_xlabel("Days since day 0")
handles = [plt.Line2D([], [], color=color, lw=2, ls=ls) for _, _, color, ls in GROUPS]
fig.legend(handles, [label for _, label, _, _ in GROUPS], title="Outage duration", loc="upper left",
           bbox_to_anchor=(0.01, 0.95), ncol=len(GROUPS), fontsize=9, title_fontsize=9, alignment="left")
fig.suptitle("Business revenue and card spending against the baseline, median county by outage duration",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(f"{OUT}/step1_fig2_business_by_outage.png"); plt.show()

# how many counties each line of Figure 2 uses
pd.Series(counts, dtype=int).rename_axis(["event", "series", "group"]).unstack("group").reindex(
    columns=[label for _, label, _, _ in GROUPS]).fillna(0).astype(int)

**Reading the two figures**

- **Figure 1**: each line is the daily median change in trips for one group of counties; the number of counties is in parentheses. If outages reduce travel, the longer a group's outage, the deeper its drop around day 0 and the slower its recovery. The gray dashed line (counties without a substantial outage) should stay near 0; if it drops clearly too, the storm itself (wind, rain, roads) has a large effect on travel, and Step 2 must control for wind and rain
- In **Figure 1**, the drop in the days before day 0 is preparation and evacuation, not the effect of outages
- **Figure 2**: weekly data are drawn as lines with dots; each dot stands for the 7 days up to that date. Womply ends in February 2022, so the top row is empty for Ian and Idalia; Affinity is weekly after June 2022. The table under the figure gives the number of counties behind each line; groups with fewer than 3 counties are not drawn
- Both figures are descriptive and do not yet show that outages **caused** the drop in travel. That test comes in Steps 2 and 3

In [ ]:
# 15. Step 1 checklist
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

n_gap_ok  = int((coverage["long_gap_peak_h"] == 0).sum())
mob_min   = float(coverage["mob_share_over_24h"].min())
val_min   = None if validation is None or validation.empty else float(validation["spearman_hours_above"].min())
net_min   = float(coverage["network_share"].min())
base_max  = float(coverage["flag_base_share_affected"].max())
rev_n     = int(coverage["revenue_over_24h"].sum())

checks = [
    ("Events without unfilled long gaps around the peak", f"{n_gap_ok} / {len(EVENTS)}", f"= {len(EVENTS)}", n_gap_ok == len(EVENTS)),
    ("Share of counties with 24+ h outages that have travel data on ≥ 90% of days, lowest event", round(mob_min, 2), "≥ 0.8", mob_min >= 0.8),
    ("Agreement with the companion project's outage-duration ranking (Spearman), lowest event", "—" if val_min is None else round(val_min, 2), "≥ 0.9", None if val_min is None else val_min >= 0.9),
    ("Share of study-area counties present in the 2019 network, lowest event", round(net_min, 3), "≥ 0.95", net_min >= 0.95),
    ("Share of affected counties with a contaminated baseline, highest event", round(base_max, 2), "≤ 0.25", base_max <= 0.25),
    ("Counties with Womply data and 24+ h outages (four events together)", rev_n, "≥ 30", rev_n >= 30),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step1_checklist.csv", index=False)
check_tab